# 3장. 요즘 트랜스포머의 부품을 하나씩 바꾸기

2장의 모델에서 위치 정보(RoPE), 정규화(RMSNorm), 피드포워드 층(SwiGLU), 어텐션의 키와 값(그룹 질의 어텐션)을 하나씩 바꾸고, 바꿀 때마다 글자당 비트와 속도를 잰다.

**준비:** NSMC의 `ratings_train.txt`를 왼쪽 파일 창에 올리고, 코랩에서는 `!pip install tokenizers`를 먼저 실행한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 58분쯤 걸렸다.

## 3.1 기준 모델을 다시 세우기

In [ ]:
import math, time, os, sys, platform
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
os.environ["RAYON_NUM_THREADS"] = "2"                # tokenizers도 스레드 2개만 쓰게 한다
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

torch.set_num_threads(2)
raw = pd.read_csv("ratings_train.txt", sep="\t").dropna(subset=["document"])
raw = raw.reset_index(drop=True)
order = np.random.default_rng(0).permutation(len(raw))   # 1, 2장과 같은 나누기
texts_tr = raw.iloc[order[:30000]]["document"].tolist()
texts_va = raw.iloc[order[-10000:]]["document"].tolist()
texts_un = raw.iloc[order[30000:-10000]]["document"].tolist()

counts = Counter(ch for text in texts_tr for ch in text)
letters = {ch for ch, n in counts.items() if n >= 5}
UNK_CH, SP = "□", "▁"
clean = lambda text: "".join(ch if ch in letters else UNK_CH for ch in text[:64]).replace(" ", SP)
un, va, dev = [clean(t) for t in texts_un], [clean(t) for t in texts_va], [clean(t) for t in texts_tr[:5000]]
C_va, C_dev = sum(len(t) + 1 for t in va), sum(len(t) + 1 for t in dev)

# 2장의 글자 BPE, 어휘 4,000개
tok = Tokenizer(models.BPE(unk_token=UNK_CH))
tok.pre_tokenizer = pre_tokenizers.Split(SP, behavior="merged_with_next")
tok.train_from_iterator(un, trainers.BpeTrainer(
    vocab_size=4000, show_progress=False, initial_alphabet=sorted(letters - {" "} | {SP, UNK_CH}),
    special_tokens=["<패딩>", "<시작>", "<끝>"]))
PAD, START, END = 0, 1, 2

def to_tensor(texts):
    """<시작> + 토큰 번호 + <끝>. 남는 칸은 0(패딩)이다."""
    rows = [[START] + e.ids + [END] for e in tok.encode_batch(texts)]
    X = torch.zeros(len(rows), max(map(len, rows)), dtype=torch.long)
    for i, r in enumerate(rows):
        X[i, :len(r)] = torch.tensor(r)
    return X

X_un, X_dev, X_va = to_tensor(un), to_tensor(dev), to_tensor(va)
print(tok.get_vocab_size(), tuple(X_un.shape), round((X_va != PAD).sum().item() / len(va) - 2, 2), C_va)
print(sys.version.split()[0], torch.__version__, tokenizers.__version__,
      platform.machine(), torch.get_num_threads())

In [ ]:
def batches(lengths, seed, size=64, chunk=50):
    """길이가 비슷한 문장 64개씩 배치를 만든다(1, 2장과 같다)."""
    shuffler = torch.Generator().manual_seed(seed)
    mixed = torch.randperm(len(lengths), generator=shuffler)
    made = []
    for start in range(0, len(mixed), size * chunk):
        part = mixed[start:start + size * chunk]
        part = part[torch.argsort(lengths[part])]
        made += [part[k:k + size] for k in range(0, len(part), size)]
    return [made[k] for k in torch.randperm(len(made), generator=shuffler)]

def loss_sum(net, data):
    """목표 토큰의 손실 합(자연로그)과 목표 토큰 수."""
    net.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for k in range(0, len(data), 500):
            ids = data[k:k + 500]
            out = net(ids[:, :-1])
            total += F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                                     ignore_index=PAD, reduction="sum").item()
            n += (ids[:, 1:] != PAD).sum().item()
    return total, n

def fit(net, X, lr, seed, epochs):
    """1, 2장의 fit과 같다. AdamW, 배치 64, 처음 10% 걸음 동안 학습률을 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    lengths = (X != PAD).sum(dim=1)
    total, step = epochs * math.ceil(len(X) / 64), 0
    warm = max(1, total // 10)
    for epoch in range(epochs):
        net.train()
        for rows in batches(lengths, seed * 1000 + epoch):
            step += 1
            wave = 0.5 * (1 + math.cos(math.pi * step / total))
            for group in optimizer.param_groups:
                group["lr"] = lr * min(1, step / warm) * wave
            ids = X[rows][:, :int(lengths[rows].max())]
            optimizer.zero_grad()
            out = net(ids[:, :-1])
            F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                            ignore_index=PAD).backward()
            optimizer.step()
    return net

bpc = lambda net, data, chars: loss_sum(net, data)[0] / math.log(2) / chars   # 글자당 비트
count = lambda net: sum(p.numel() for p in net.parameters())
align = lambda text, width: text + " " * max(width - len(text) - sum(ord(c) > 0x1100 for c in text), 1)   # 한글 폭 맞추기

In [ ]:
class Attention(nn.Module):
    """셀프 어텐션. rope=True면 RoPE(3.2절), kv_heads가 heads보다 작으면 그룹 질의 어텐션(3.5절)."""
    def __init__(self, dim, heads, kv_heads=None, rope=False, length=66):
        super().__init__()
        self.heads, self.kv_heads, self.hd = heads, kv_heads or heads, dim // heads
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, self.kv_heads * self.hd)
        self.to_v = nn.Linear(dim, self.kv_heads * self.hd)
        self.out = nn.Linear(dim, dim)
        self.rope = rope
        if rope:
            cos, sin = rope_tables(length, self.hd)        # 3.2절에서 만든다
            self.register_buffer("cos", cos)
            self.register_buffer("sin", sin)

    def forward(self, x, hide):
        B, L, D = x.shape
        q = self.to_q(x).view(B, L, self.heads, self.hd).transpose(1, 2)
        k = self.to_k(x).view(B, L, self.kv_heads, self.hd).transpose(1, 2)
        v = self.to_v(x).view(B, L, self.kv_heads, self.hd).transpose(1, 2)
        if self.rope:
            q, k = rotate(q, self.cos, self.sin), rotate(k, self.cos, self.sin)
        if self.kv_heads < self.heads:                        # 키와 값 머리 하나를 질의 머리 여럿이 함께 쓴다
            k = k.repeat_interleave(self.heads // self.kv_heads, dim=1)
            v = v.repeat_interleave(self.heads // self.kv_heads, dim=1)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.hd)
        scores = scores.masked_fill(hide[:, None, None, :], float("-inf"))
        later = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(later, float("-inf"))
        return self.out((torch.softmax(scores, dim=-1) @ v).transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, dim, heads, cfg):
        super().__init__()
        Norm = RMSNorm if cfg.get("norm") == "rms" else nn.LayerNorm          # RMSNorm은 3.3절
        self.norm1, self.norm2 = Norm(dim), Norm(dim)
        self.attend = Attention(dim, heads, cfg.get("kv"), cfg.get("rope", False))
        if cfg.get("mlp"):                                                      # SwiGLU는 3.4절
            self.mlp = SwiGLU(dim, cfg["mlp"])
        else:
            self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x, hide):
        x = x + self.attend(self.norm1(x), hide)
        return x + self.mlp(self.norm2(x))

class LanguageModel(nn.Module):
    """2장의 모델. cfg로 이 장에서 바꿀 네 곳을 고른다(빈 사전이면 2장과 같다)."""
    def __init__(self, cfg, size=4000, dim=64, heads=4, layers=2, length=66):
        super().__init__()
        self.table = nn.Embedding(size, dim)
        self.where = None if cfg.get("rope") else nn.Embedding(length, dim)   # RoPE면 위치 표가 없다
        self.blocks = nn.ModuleList(Block(dim, heads, cfg) for _ in range(layers))
        self.norm = RMSNorm(dim) if cfg.get("norm") == "rms" else nn.LayerNorm(dim)
        self.head = nn.Linear(dim, size)

    def forward(self, ids):
        x = self.table(ids)
        if self.where is not None:
            x = x + self.where(torch.arange(ids.shape[1]))
        hide = ids == PAD
        for block in self.blocks:
            x = block(x, hide)
        return self.head(self.norm(x))

torch.manual_seed(0)
print(count(LanguageModel({})))

In [ ]:
RATES = [0.003, 0.01, 0.03]
results, nets = {}, {}

def one_run(cfg, lr, seed):
    torch.manual_seed(seed)
    net = LanguageModel(cfg)
    lap = time.perf_counter()
    fit(net, X_un, lr, seed, epochs=4)
    sec = (time.perf_counter() - lap) / 4                       # 1에폭 학습 시간
    return {"dev": bpc(net, X_dev, C_dev), "va": bpc(net, X_va, C_va), "sec": sec, "net": net}

def compare(name, cfg, rates=RATES):
    """학습률마다 시드 0으로 학습시켜 개발 문장으로 고르고, 고른 학습률로 시드 1, 2를 더 돌린다."""
    runs = {(lr, 0): one_run(cfg, lr, 0) for lr in rates}
    best = min(rates, key=lambda lr: runs[lr, 0]["dev"])
    for seed in [1, 2]:
        runs[best, seed] = one_run(cfg, best, seed)
    va = [runs[best, s]["va"] for s in range(3)]
    nets[name] = runs[best, 0]["net"]
    lap = time.perf_counter()
    loss_sum(nets[name], X_va)                                  # 검증 문장 10,000개를 한 번 읽는 시간
    results[name] = {"cfg": cfg, "best": best, "dev": {lr: runs[lr, 0]["dev"] for lr in rates},
                     "va0": {lr: runs[lr, 0]["va"] for lr in rates},
                     "va": va, "med": float(np.median(va)), "params": count(nets[name]),
                     "sec": float(np.median([runs[best, s]["sec"] for s in range(3)])),
                     "read": time.perf_counter() - lap}
    r = results[name]
    print(f"{align(name, 14)}파라미터 {r['params']:>7,}  개발 BPC " + " ".join(f"{lr}:{v:.4f}" for lr, v in r["dev"].items())
          + f"  → 학습률 {best}, 검증 BPC {r['med']:.4f} ({min(va):.4f}~{max(va):.4f}),"
          + f" 1에폭 {r['sec']:.1f}초, 읽기 {r['read']:.1f}초", flush=True)

compare("기준", {})
print(f"2장과 같은 설정(학습률 0.01, 시드 0)의 검증 BPC {results['기준']['va0'][0.01]:.4f}")

## 3.2 위치 정보: 학습하는 위치 표 대신 RoPE

In [ ]:
def rope_tables(length, hd, base=10000.0):
    """위치 m과 짝 i의 회전각 m·θ_i의 cos, sin 표. θ_i = base^(-2i/hd)."""
    theta = base ** (-torch.arange(0, hd, 2).float() / hd)               # 짝마다 도는 빠르기(hd/2개)
    angle = torch.arange(length).float()[:, None] * theta[None, :]      # (길이, hd/2)
    return torch.cos(angle), torch.sin(angle)

def rotate(x, cos, sin):
    """x의 이웃한 두 값을 한 짝으로 보고, 위치에 따른 각도만큼 돌린다. x: (배치, 머리, 길이, hd)."""
    a, b = x[..., 0::2], x[..., 1::2]
    c, s = cos[:x.shape[-2]], sin[:x.shape[-2]]
    return torch.stack([a * c - b * s, a * s + b * c], dim=-1).flatten(-2)

cos, sin = rope_tables(66, 16)
torch.manual_seed(0)
q, k = torch.randn(16), torch.randn(16)
place = lambda x, m: rotate(x.expand(1, 1, m + 1, 16), cos, sin)[0, 0, m]   # 위치 m에 놓인 벡터를 돌린다
for m, n in [(3, 1), (10, 8), (60, 58), (40, 10), (31, 1)]:
    print(f"질의 위치 {m:>2}, 키 위치 {n:>2}, 거리 {m - n:>2}: 점수 {place(q, m) @ place(k, n):.6f}")
print(f"돌리기 전후의 길이 {q.norm():.6f} {place(q, 37).norm():.6f}")
# 같은 회전을 복소수 곱셈으로 다시 계산해 맞춘다
z = torch.view_as_complex(q.view(8, 2).contiguous())
turn = torch.polar(torch.ones(8), torch.atan2(sin[37], cos[37]))
print(f"복소수 곱셈과의 차이 {(torch.view_as_real(z * turn).flatten() - place(q, 37)).abs().max():.2e}")

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for path in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf"))
             + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(path)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 9, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"

N = 40
score = np.array([[float(place(q, m) @ place(k, n)) if n <= m else np.nan for n in range(N)] for m in range(N)])
table = nets["기준"].where.weight.detach()[:N]
unit = table / table.norm(dim=1, keepdim=True)
similar = (unit @ unit.T).numpy()
fig, axes = plt.subplots(2, 1, figsize=(5.2, 6.8), dpi=260)
for ax, data, title, cmap, label in [
        (axes[0], score, "① RoPE: 같은 q, k를 위치만 바꿔 놓았을 때의 점수 q·k", "PuOr", "점수"),
        (axes[1], similar, "② 기준 모델이 학습한 위치 표: 두 위치 벡터의 코사인 유사도", "RdBu_r", "코사인 유사도")]:
    lim = np.nanmax(np.abs(data))
    im = ax.imshow(data, cmap=cmap, vmin=-lim, vmax=lim, origin="upper")
    ax.set_title(title, fontsize=8.8, loc="left", color="#374151")
    ax.set_xlabel("키의 위치 n" if ax is axes[0] else "위치")
    ax.set_ylabel("질의의 위치 m" if ax is axes[0] else "위치")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).set_label(label, fontsize=8.4)
axes[0].annotate("거리 m−n이 같은 칸(대각선)은\n점수가 모두 같다", (8, 18), xytext=(16, 3.8), textcoords="data",
                 fontsize=8.4, color="#111827", arrowprops=dict(arrowstyle="->", color="#111827", lw=0.8),
                 bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.5))
for (m, n), dx, dy in [((3, 1), 3, 5), ((10, 8), 10, 0), ((31, 1), 7, 4)]:     # 위 셀에서 출력한 쌍
    axes[0].add_patch(plt.Circle((n, m), 0.85, fill=False, ec=GREEN, lw=1.3))
    axes[0].annotate(f"({m}, {n}) 거리 {m - n}\n점수 {score[m, n]:.2f}", (n, m), xytext=(n + dx, m + dy), fontsize=8,
                     color="#0F7A57", arrowprops=dict(arrowstyle="-", color=GREEN, lw=0.8),
                     bbox=dict(fc="white", ec=GREEN, lw=0.6, pad=1.2))
fig.tight_layout(); fig.savefig("fig_3_rope.png"); plt.close(fig)

In [ ]:
compare("RoPE", {"rope": True})

## 3.3 정규화: LayerNorm 대신 RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    """평균을 빼지 않고, 제곱 평균의 제곱근(RMS)으로만 나눈 다음 학습하는 배율을 곱한다."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps) * self.weight

torch.manual_seed(0)
x = torch.randn(4, 9, 64) * 3 + 2                 # 평균이 2쯤, 퍼짐이 3쯤인 값
mine, lib, layer = RMSNorm(64)(x), nn.RMSNorm(64, eps=1e-6)(x), nn.LayerNorm(64)(x)
print(f"nn.RMSNorm과의 차이 {(mine - lib).abs().max():.2e}")
for name, y in [("입력", x), ("LayerNorm", layer), ("RMSNorm", mine)]:
    print(f"{name:<10} 평균 {y.mean(-1).mean():.4f}  RMS {y.pow(2).mean(-1).sqrt().mean():.4f}")
print("정규화 층 하나의 파라미터", sum(p.numel() for p in nn.LayerNorm(64).parameters()),
      sum(p.numel() for p in RMSNorm(64).parameters()))

In [ ]:
compare("RMSNorm", {"norm": "rms"})

## 3.4 피드포워드 층: GELU 대신 SwiGLU

In [ ]:
class SwiGLU(nn.Module):
    """64를 두 번 따로 hidden으로 늘려, SiLU를 거친 쪽을 문으로 삼아 다른 쪽에 곱한 다음 다시 줄인다."""
    def __init__(self, dim, hidden):
        super().__init__()
        self.gate = nn.Linear(dim, hidden)
        self.up = nn.Linear(dim, hidden)
        self.down = nn.Linear(hidden, dim)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

z = torch.linspace(-4, 4, 9)
print("SiLU", [round(v, 3) for v in F.silu(z).tolist()])
print(f"x·sigmoid(x)와의 차이 {(F.silu(z) - z * torch.sigmoid(z)).abs().max():.2e}")
gelu_mlp = 64 * 256 + 256 + 256 * 64 + 64
for h in [170, 256]:
    print(f"숨은 크기 {h}: SwiGLU 파라미터 {sum(p.numel() for p in SwiGLU(64, h).parameters()):,}, GELU 층 {gelu_mlp:,}")

In [ ]:
compare("SwiGLU(170)", {"mlp": 170})
compare("SwiGLU(256)", {"mlp": 256}, rates=[results["SwiGLU(170)"]["best"]])

## 3.5 어텐션: 키와 값을 나눠 쓰는 그룹 질의 어텐션

In [ ]:
torch.manual_seed(0)
att = Attention(64, 4, kv_heads=2)
xs = torch.randn(2, 7, 64)
ours = att(xs, torch.zeros(2, 7, dtype=torch.bool))
# 같은 가중치로 질의·키·값을 만들어 PyTorch의 scaled_dot_product_attention(enable_gqa=True)에 넣는다
qg = att.to_q(xs).view(2, 7, 4, 16).transpose(1, 2)
kg = att.to_k(xs).view(2, 7, 2, 16).transpose(1, 2)
vg = att.to_v(xs).view(2, 7, 2, 16).transpose(1, 2)
ref = att.out(F.scaled_dot_product_attention(qg, kg, vg, is_causal=True, enable_gqa=True).transpose(1, 2).reshape(2, 7, 64))
print(f"scaled_dot_product_attention과의 차이 {(ours - ref).abs().max():.2e}")
print("키·값 머리  키·값 층 파라미터(블록 하나)  토큰 하나의 KV 캐시(바이트)")
for kv in [4, 2, 1]:
    a = Attention(64, 4, kv_heads=kv)
    kv_params = sum(p.numel() for p in [*a.to_k.parameters(), *a.to_v.parameters()])
    cache = 2 * 2 * kv * 16 * 4          # 키와 값 × 블록 2개 × 머리 × 머리 차원 16 × float32 4바이트
    print(f"{kv:>9}  {kv_params:>24,}  {cache:>24,}")

In [ ]:
compare("GQA(kv 2)", {"kv": 2})
compare("MQA(kv 1)", {"kv": 1}, rates=[results["GQA(kv 2)"]["best"]])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.6), dpi=260, gridspec_kw={"height_ratios": [1.45, 1]})
ax = axes[0]
kv_colors = [BLUE, ORANGE, GREEN, PURPLE]
for row, (kv, name) in enumerate([(4, "멀티헤드(MHA): 키·값 머리 4"), (2, "그룹 질의(GQA): 키·값 머리 2"), (1, "멀티쿼리(MQA): 키·값 머리 1")]):
    y = 2 - row
    ax.text(-0.2, y + 0.36, name, fontsize=8.6, fontweight="bold", color="#374151", va="center")
    for h in range(4):
        g = h // (4 // kv)
        ax.add_patch(plt.Rectangle((h * 1.15, y - 0.1), 0.95, 0.3, fc="white", ec=kv_colors[g], lw=1.2))
        ax.text(h * 1.15 + 0.475, y + 0.05, f"질의 {h + 1}", ha="center", va="center", fontsize=8.2)
    for j in range(kv):
        span = 4 // kv
        x0, x1 = j * span * 1.15, (j * span + span - 1) * 1.15 + 0.95
        ax.add_patch(plt.Rectangle((x0, y - 0.48), x1 - x0, 0.28, fc=kv_colors[j], ec="none", alpha=0.9))
        ax.text((x0 + x1) / 2, y - 0.34, f"키·값 {j + 1}", ha="center", va="center", fontsize=8.2, color="white", fontweight="bold")
ax.set_xlim(-0.25, 4.7); ax.set_ylim(-0.62, 2.62); ax.axis("off")
ax.set_title("① 같은 색의 질의 머리는 키·값 머리 하나를 함께 쓴다", fontsize=8.8, loc="left", color="#374151")
ax = axes[1]
caches = [2 * 2 * kv * 16 * 4 for kv in [4, 2, 1]]
bars = ax.barh(["MHA(4)", "GQA(2)", "MQA(1)"], caches, color=GRAY, height=0.62)
for b, c, kv in zip(bars, caches, [4, 2, 1]):
    p = 2 * (64 * kv * 16 + kv * 16)                                   # 키·값 층의 가중치와 편향(블록 하나)
    ax.annotate(f"{c:,}바이트\n(키·값 층 파라미터 {p:,}개/블록)", (c, b.get_y() + b.get_height() / 2), xytext=(4, 0),
                textcoords="offset points", va="center", fontsize=8.2, linespacing=1.25)
ax.invert_yaxis(); ax.set_xlim(0, 1900); ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("토큰 하나를 만들 때 기억해 두는 키와 값(KV 캐시, 바이트)")
ax.set_title("② 이 장의 모델(블록 2개, 머리 차원 16, float32)에서\n토큰 하나의 KV 캐시", fontsize=8.8, loc="left", color="#374151")
fig.tight_layout(); fig.savefig("fig_3_gqa.png"); plt.close(fig)

## 3.6 네 부품을 함께 바꾸기

In [ ]:
compare("넷 다", {"rope": True, "norm": "rms", "mlp": 170, "kv": 2})
compare("SwiGLU 뺀 셋", {"rope": True, "norm": "rms", "kv": 2})

In [ ]:
base = results["기준"]
print("설정          파라미터  학습률  검증 BPC(중앙값, 범위)   기준과 차이  1에폭  읽기")
for name, r in results.items():
    gap = r["med"] - base["med"]
    clear = " *" if (max(r["va"]) < min(base["va"]) or min(r["va"]) > max(base["va"])) and name != "기준" else ""
    print(f"{align(name, 13)}{r['params']:>8,}  {r['best']:>5}  {r['med']:.4f} ({min(r['va']):.4f}~{max(r['va']):.4f})"
          f"  {gap:+.4f}  {r['sec']:>5.1f}  {r['read']:>4.1f}{clear}")
print("* 기준 모델과 범위가 겹치지 않음. 시간은 초")

In [ ]:
names = list(results)
fig, axes = plt.subplots(2, 1, figsize=(5.2, 6.4), dpi=260, gridspec_kw={"height_ratios": [1.45, 1]})
ax = axes[0]
lo_b, hi_b = min(base["va"]), max(base["va"])
ax.axvspan(lo_b, hi_b, color=GRAY, alpha=0.25, lw=0)
for i, n in enumerate(names):
    r = results[n]
    better, worse = max(r["va"]) < lo_b, min(r["va"]) > hi_b
    color = GREEN if better else (RED if worse else BLUE)
    ax.plot([min(r["va"]), max(r["va"])], [i, i], color=color, lw=2.0, solid_capstyle="round")
    ax.plot([r["med"]], [i], "o", color=color, ms=5.5)
    ax.text(max(r["va"]), i - 0.32, f"{r['med']:.4f}", va="bottom", ha="right", fontsize=8, color=color)
every = [v for n in names for v in results[n]["va"]]
ax.set_xlim(min(every) - 0.006, max(every) + 0.006)
ax.set_ylim(len(names) - 0.4, -0.8)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8.6)
ax.set_xlabel("검증 문장의 글자당 비트\n(점은 시드 3개의 중앙값, 선은 범위)")
ax.set_title("① 회색 띠는 기준 모델의 범위.\n초록은 띠보다 낮고, 빨강은 높고, 파랑은 겹친다", fontsize=8.6, loc="left", color="#374151")
ax.grid(axis="x", color="#e5e7eb", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
ax = axes[1]
ax.barh(range(len(names)), [results[n]["sec"] for n in names], color=GRAY, height=0.6)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8.6); ax.invert_yaxis()
ax.set_xlim(0, max(results[n]["sec"] for n in names) * 1.18)
for i, n in enumerate(names):
    ax.text(results[n]["sec"], i, f" {results[n]['sec']:.1f}초", va="center", fontsize=8.2)
ax.set_xlabel("1에폭 학습 시간(초, 시드 3개의 중앙값)")
ax.set_title("② 학습 시간", fontsize=8.6, loc="left", color="#374151")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_3_results.png"); plt.close(fig)